# BA283 A/B — FiLM vs plain GRU (concat)
Reads the expanded dataset straight from Google Drive (staged by the box).
Trains both conditionings on identical data, compares val ESR, exports both.


## 1. Mount Google Drive via rclone
VFS full-cache: first epoch (and the warm step below) pulls WAVs from Drive + caches them on Colab's local disk; every later epoch reads the local copy (no per-file FUSE latency). Dataset is small (~hundreds of MB) so the whole set stays cached.

Run the cell, then **upload your `rclone.conf`** when prompted (find it via `rclone config file`). It holds the Drive token, so it's written straight to disk and never printed.


In [ ]:
import os, glob, subprocess
from google.colab import files

# install rclone + fuse (no-op if present)
if subprocess.run(['which','rclone'],capture_output=True).returncode:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash', shell=True, check=True)
subprocess.run('apt-get -qq install -y fuse3 || apt-get -qq install -y fuse', shell=True)

# upload rclone.conf (token written to disk, not echoed)
cfg = os.path.expanduser('~/.config/rclone/rclone.conf')
os.makedirs(os.path.dirname(cfg), exist_ok=True)
up = files.upload()  # pick your rclone.conf
assert up, 'no file uploaded'
open(cfg,'wb').write(next(iter(up.values())))
REMOTE = subprocess.run(['rclone','listremotes'],capture_output=True,text=True).stdout.split()[0]
print('remote:', REMOTE)

# mount the dataset folder directly. NOTE: with --daemon, DO NOT capture stdout/
# stderr -- the forked daemon inherits the pipes and run() would hang on EOF that
# never comes. Send them to DEVNULL; diagnostics go to the --log-file.
MNT = '/content/drive'; os.makedirs(MNT, exist_ok=True)
LOG = '/content/rclone_mount.log'
if not os.path.ismount(MNT):
    r = subprocess.run(['rclone','mount', REMOTE+'ba283_spice_gain', MNT,
        '--vfs-cache-mode','full',      # cache whole files locally -> fast re-reads
        '--vfs-cache-max-size','12G',   # > dataset size, nothing evicted mid-run
        '--vfs-cache-max-age','9999h',
        '--vfs-read-chunk-size','16M',  # big sequential pulls for WAVs
        '--buffer-size','32M',
        '--dir-cache-time','9999h',     # static dataset -> never re-list
        '--attr-timeout','1h',
        '--poll-interval','0',          # no remote change polling
        '--daemon','--log-file',LOG,'--log-level','INFO'],
        stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if r.returncode:
        print(open(LOG).read()[-2000:] if os.path.exists(LOG) else '(no log)')
        raise RuntimeError('rclone mount failed (see log above)')
print('mounted at', MNT)

DATA = MNT  # mount root IS the ba283_spice_gain folder
print('DATA =', DATA)
assert os.path.exists(os.path.join(DATA,'manifest.csv')), 'manifest.csv missing (batch not done?)'

# warm the VFS cache: read every file THROUGH the mount so rclone caches it locally
subprocess.run(f"find {DATA!r} -type f -print0 | xargs -0 -P8 cat > /dev/null", shell=True)
print('rows:', sum(1 for _ in open(os.path.join(DATA,'train.csv')))-1, 'train')


## 2. Get the training code + deps
Paste a GitHub token with read access (kept out of output via getpass).


In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ')
import subprocess
subprocess.run(['git','clone','-q','https://x-access-token:%s@github.com/gustavokch/ba283-vst3.git'%TOKEN,'/content/ba283'])
%cd /content/ba283
!git pull -q
!pip -q install soundfile pandas tensorboard scipy
import torch; print('cuda:', torch.cuda.is_available())


## 3. Rewrite manifest paths to the Colab dataset location
The box wrote absolute paths; remap each to DATA/<wet|dry>/<basename>.


In [ ]:
import pandas as pd, os
for name in ['train','val','test']:
    df = pd.read_csv(os.path.join(DATA, name+'.csv'))
    for col in ['dry_path','wet_path']:
        df[col] = df[col].apply(lambda x: os.path.join(DATA, os.path.basename(os.path.dirname(x)), os.path.basename(x)))
    df.to_csv('/content/%s.csv'%name, index=False)
print(pd.read_csv('/content/train.csv').iloc[0].to_dict())
assert os.path.exists(pd.read_csv('/content/train.csv').iloc[0]['wet_path']), 'path remap failed'


## 4. Train FiLM


In [ ]:
!python -m train.train --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 1e-3 --batch 128 --stride 4096 --hidden 32 --device cuda --conditioning film \
  --checkpoint-dir ckpt_film --log-dir logs_film --patience 40 --seed 42


## 4b. Checkpoint FiLM to Drive (runtime-limit insurance)
Run right after FiLM finishes. Evals val ESR, exports the JSON + raw checkpoints, and `rclone copy`s them straight to the remote (synchronous, so they survive a runtime kill before concat finishes).


In [ ]:
import torch, numpy as np, os, glob, shutil, subprocess
from train.dataset import make_loader
from train.model import BA283LSTM
from train.export import export_rtneural_json
from train.loss import esr

def eval_val(cond, ckpt):
    m = BA283LSTM(32, conditioning=cond)
    m.load_state_dict(torch.load(ckpt+'/best.pth', map_location='cpu', weights_only=True)); m.eval()
    ld = make_loader('/content/val.csv', batch_size=32, window=4096, stride=2048,
                     num_workers=2, shuffle=False, drop_last=False)
    P=[];T=[];warm=1024
    with torch.no_grad():
        for x,y in ld:
            _,h=m(x[:,:warm]); p,_=m(x[:,warm:],h)
            P.append(p.reshape(-1)); T.append(y[:,warm:].reshape(-1))
    return m, esr(torch.cat(P),torch.cat(T)).item()

m, e_film = eval_val('film','ckpt_film')
print('film val ESR = %.4f (%.1f dB)' % (e_film, 10*np.log10(max(e_film,1e-12))))

# stage locally, then rclone copy straight to the remote (synchronous upload)
os.makedirs('/content/film_save', exist_ok=True)
export_rtneural_json(m, '/content/film_save/ba283_film.json',
                     meta={'source':'colab-ab','conditioning':'film','val_esr':e_film})
for f in glob.glob('ckpt_film/*.pth'):
    shutil.copy(f, '/content/film_save/film_'+os.path.basename(f))
open('/content/film_save/film_val_esr.txt','w').write('%.6f\n' % e_film)
REMOTE = globals().get('REMOTE','gdrive:')
subprocess.run(['rclone','copy','/content/film_save', REMOTE+'ba283_spice_gain/film_save','-P'], check=True)
print('saved -> %sba283_spice_gain/film_save' % REMOTE)


## 5. Train plain GRU (concat)


In [ ]:
!python -m train.train --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 1e-3 --batch 128 --stride 4096 --hidden 32 --device cuda --conditioning concat \
  --checkpoint-dir ckpt_concat --log-dir logs_concat --patience 40 --seed 42


## 6. Compare val ESR + export both to Drive


In [ ]:
import torch, numpy as np
from train.dataset import make_loader
from train.model import BA283LSTM
from train.export import export_rtneural_json
from train.loss import esr
def evalm(cond, ckpt):
    m = BA283LSTM(32, conditioning=cond)
    m.load_state_dict(torch.load(ckpt+'/best.pth', map_location='cpu', weights_only=True)); m.eval()
    ld = make_loader('/content/val.csv', batch_size=32, window=4096, stride=2048, num_workers=2, shuffle=False, drop_last=False)
    P=[];T=[];warm=1024
    with torch.no_grad():
        for x,y in ld:
            _,h=m(x[:,:warm]); p,_=m(x[:,warm:],h); P.append(p.reshape(-1)); T.append(y[:,warm:].reshape(-1))
    e=esr(torch.cat(P),torch.cat(T)).item()
    export_rtneural_json(m, DATA+'/ba283_%s.json'%cond, meta={'source':'colab-ab','conditioning':cond})
    return e
res = {c: evalm(c, ck) for c,ck in [('film','ckpt_film'),('concat','ckpt_concat')]}
for c,e in res.items(): print('%-7s val ESR = %.4f (%.1f dB)' % (c, e, 10*np.log10(max(e,1e-12))))
win = min(res, key=res.get); print('WINNER:', win)
print('exported ba283_film.json + ba283_concat.json to', DATA)


Bring back `ba283_<winner>.json` (on Drive). I embed it + port that conditioning to the Rust plugin (#30).
